# Can we predict failure within 90 days of an inspection?

Target: `failed_90d`. Plan: inspect data, check for leakage and time effects, build a time-ordered train/test split, compare a baseline with a few models, report honestly.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width', 140)
df = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(pd.read_csv('columns.csv').to_string(index=False))
df.head()

## 1. Data quality

In [ ]:
print(df.shape, 'missing:', df.isna().sum().sum(), 'dup rows:', df.duplicated().sum(), 'unique assets:', df.asset_tag.nunique())
print('date range', df.inspected_at.min(), '->', df.inspected_at.max())
print('failure rate', df.failed_90d.mean().round(4))
df.describe().T

Each asset appears once, so there is no repeated-asset leakage across rows.

## 2. Leakage check: `parts_replaced`
The dictionary says it counts parts replaced on the work order *that closed this inspection record*, so it is probably recorded after the outcome is known (e.g. a repair following a failure).

In [ ]:
df.groupby('parts_replaced').failed_90d.agg(['mean', 'size'])

Every record with 2+ parts replaced failed, and none with 0 did. That near-deterministic link would not be available at inspection time, so **`parts_replaced` is excluded** from the features. Section 7 shows how much it would inflate the score.

## 3. Drift over time

In [ ]:
q = df.groupby(df.inspected_at.dt.to_period('Q')).failed_90d.agg(['mean', 'size'])
display(q); q['mean'].plot.bar(title='Failure rate by inspection quarter'); plt.show()

The base rate swings a lot (about 4% -> 27% -> 3%). The last quarter may also be right-censored (failures after the data end on 2025-12-30 may not be observed yet). So I use a **chronological** split rather than a random one, which is closer to real deployment.

## 4. Features vs. outcome

In [ ]:
num = ['vibration_rms', 'bearing_temp_c', 'ambient_temp_c', 'meter_hours', 'load_pct']
fig, ax = plt.subplots(1, 5, figsize=(18, 3))
for a, c in zip(ax, num):
    b = pd.qcut(df[c], 10, duplicates='drop')
    df.groupby(b, observed=True).failed_90d.mean().plot(ax=a, marker='o'); a.set_title(c); a.set_xticks([])
plt.tight_layout(); plt.show()
print(df.groupby('site').failed_90d.mean().round(3).to_dict())

## 5. Models
Features: five measurements, site, and a derived `temp_rise` (bearing minus ambient). Train on the first 70% in time, test on the last 30%. CV inside train uses time-ordered folds.

In [ ]:
df['temp_rise'] = df.bearing_temp_c - df.ambient_temp_c
num_f = num + ['temp_rise']; cat_f = ['site']
X = df[num_f + cat_f]; y = df.failed_90d
cut = int(len(df) * .7)
Xtr, Xte, ytr, yte = X[:cut], X[cut:], y[:cut], y[cut:]
print('train rate', ytr.mean().round(3), 'test rate', yte.mean().round(3), '| test starts', df.inspected_at[cut])
pre = ColumnTransformer([('n', StandardScaler(), num_f), ('c', OneHotEncoder(), cat_f)])
models = {'baseline (prior)': DummyClassifier(),
          'logistic': make_pipeline(pre, LogisticRegression(max_iter=1000)),
          'grad. boosting': make_pipeline(pre, HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=200, random_state=0))}
rows = []; fit = {}
for n, m in models.items():
    cv = .5 if n.startswith('base') else cross_val_score(m, Xtr, ytr, cv=TimeSeriesSplit(5), scoring='roc_auc').mean()
    m.fit(Xtr, ytr); p = m.predict_proba(Xte)[:, 1]; fit[n] = p
    rows.append(dict(model=n, cv_auc_train=cv, test_auc=roc_auc_score(yte, p), test_avg_precision=average_precision_score(yte, p)))
pd.DataFrame(rows).round(3).set_index('model')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for n, p in fit.items():
    if n.startswith('base'): continue
    f, t, _ = roc_curve(yte, p); ax[0].plot(f, t, label=n)
    pr, rc, _ = precision_recall_curve(yte, p); ax[1].plot(rc, pr, label=n)
ax[0].plot([0, 1], [0, 1], 'k--'); ax[0].set(title='ROC', xlabel='FPR', ylabel='TPR')
ax[1].axhline(yte.mean(), color='k', ls='--'); ax[1].set(title='Precision-recall', xlabel='recall', ylabel='precision'); ax[0].legend(); plt.show()

## 6. What drives the prediction?

In [ ]:
m = models['grad. boosting']
r = permutation_importance(m, Xte, yte, scoring='roc_auc', n_repeats=10, random_state=0)
pd.Series(r.importances_mean, index=X.columns).sort_values().plot.barh(title='Permutation importance (test AUC drop)'); plt.show()
lr = models['logistic']; names = num_f + list(lr[0].named_transformers_['c'].get_feature_names_out())
print(pd.Series(lr[-1].coef_[0], index=names).round(3).sort_values())

## 7. Sensitivity checks

In [ ]:
# (a) What leakage looks like: add parts_replaced
Xl = df[num_f + cat_f + ['parts_replaced']]
pre_l = ColumnTransformer([('n', StandardScaler(), num_f + ['parts_replaced']), ('c', OneHotEncoder(), cat_f)])
ml = make_pipeline(pre_l, HistGradientBoostingClassifier(max_depth=3, random_state=0)).fit(Xl[:cut], ytr)
print('AUC WITH parts_replaced (leaky, do not use):', round(roc_auc_score(yte, ml.predict_proba(Xl[cut:])[:, 1]), 3))
# (b) stability across test quarters
t = df.iloc[cut:].assign(p=fit['grad. boosting'])
display(t.groupby(t.inspected_at.dt.to_period('Q')).apply(lambda g: pd.Series(dict(n=len(g), rate=g.failed_90d.mean(), auc=roc_auc_score(g.failed_90d, g.p) if g.failed_90d.nunique() > 1 else np.nan))).round(3))
# (c) calibration of predicted risk on test
t['bin'] = pd.qcut(t.p, 5, duplicates='drop')
display(t.groupby('bin', observed=True).agg(pred=('p', 'mean'), actual=('failed_90d', 'mean'), n=('p', 'size')).round(3))

## 8. Conclusions
- **Yes, but only moderately.** Using inspection-time measurements, gradient boosting reaches test ROC AUC of about 0.75 (logistic about 0.68) against 0.5 for the baseline. Average precision is about 0.15 vs a 0.056 base rate, roughly 2.6x lift. Useful for ranking inspections by risk, not for confident individual calls.
- **Drivers:** vibration RMS, then load, then bearing temperature (and temperature rise over ambient). Higher meter hours and ambient temperature point the other way in the logistic model. Site carries essentially no signal.
- **`parts_replaced` is leakage.** It is recorded on the work order that closes the inspection, and gives AUC about 0.998 when included. Don't use it as a predictor.
- **Caveats:** the failure rate drifts heavily by quarter (4% to 27% to 3%), so predicted probabilities are over-confident on the recent test period (calibration table); use the model for ranking or recalibrate. The final quarter may be right-censored. One inspection per asset means no per-asset history features were possible. The test set has only about 100 positives, so the AUC estimate is noisy (cross-validated AUC 0.72-0.74 agrees).
- **Next steps:** confirm how `parts_replaced` is recorded and whether the late-2025 drop is censoring; check whether the 2024 Q2-Q3 spike has a known cause (a batch, a season, a policy change); add per-asset history if repeat inspections exist.